(Tutorial_Get_distance_clusters)=
# Getting distance clusters

*Grouping a bounded population using a declared physical cutoff.*

This experimental tool accepts a finite nonnegative symmetric length matrix with
zero diagonal. You supply the physical units; an untyped numeric array is rejected.
The matrix can come from any supported geometric analysis. A triangle inequality
is not required. Missing or directional distances are never repaired.

:::{admonition} API documentation
:class: dropdown
{func}`molsysmt.structure.get_distance_clusters`
:::

:::{versionadded} 1.0.0
:::

## Distinguishing linkage rules

The inclusive rule is `linkage_distance <= cutoff_distance`. Complete linkage uses
the largest cross-cluster member distance and bounds each cluster's diameter.
Single linkage uses the smallest cross distance and can chain distant members.
This analytic chain makes that distinction visible.


In [1]:
import numpy as np
import molsysmt as msm
from molsysmt.native import Structures
puw = msm.pyunitwizard
distances = puw.quantity([[0., 1., 2.], [1., 0., 1.], [2., 1., 0.]], 'nm')
complete = msm.structure.get_distance_clusters(distances, '1 nm')
single = msm.structure.get_distance_clusters(distances, '1 nm', method='single_linkage')
print('Complete:', complete['cluster_indices'].tolist())
print('Single:', single['cluster_indices'].tolist())
print('Single diameter (nm):', puw.get_value(single['cluster_diameters'], to_unit='nm'))
assert complete['cluster_indices'].tolist() == [0, 0, 1]
assert single['cluster_indices'].tolist() == [0, 0, 0]


Complete: [0, 0, 1]
Single: [0, 0, 0]
Single diameter (nm): [2.]


## Retaining members and representatives

`cluster_indices` is aligned to the original matrix order. Slice `member_indices`
between adjacent `cluster_offsets` to recover each population. Clusters and members
are sorted by earliest input position. Exact merge ties choose the earliest pair
of smallest original cluster members; tied partitions may depend on input order.

The default medoid minimizes the sum of distances within its cluster, with exact
ties resolved by earliest position. `representative='first'` chooses the earliest
member directly. Neither rule examines docking scores or biological relevance.


In [2]:
for start, stop, representative in zip(single['cluster_offsets'][:-1],
                                      single['cluster_offsets'][1:],
                                      single['representative_indices']):
    print('Members:', single['member_indices'][start:stop].tolist(),
          'representative:', int(representative))
assert single['representative_indices'].tolist() == [1]


Members: [0, 1, 2] representative: 1


## Knowing the method and limits

The implementation follows the primitive agglomerative definition and min/max
updates in [Müllner (2011), sections 2.1 and 2.4, figures 1–2](https://arxiv.org/abs/1109.2378).
It does not implement that paper's optimized algorithms or execute SciPy.
The descriptive method names identify the rule, not a software brand.
Detached references and optional Ackredit retain this distinction.

The bounded implementation uses quadratic numeric storage and cubic scan work.
`max_work` bounds `n_members**2 * (n_members - 1)`; configured numeric RAM checks
apply before canonical conversion/copies. Caller storage, Python objects and total
RSS are excluded. Empty input returns offsets `[0]`; duplicates may merge at zero.

Cutoffs compare computed float64 nm values exactly, without hidden rounding or
tolerance. Unit conversion and geometric arithmetic can place a value just above
an intended mathematical boundary. Declare any scientifically justified margin in
your cutoff, rather than relying on an undocumented tolerance. `matrix_sha256`
identifies represented canonical numeric values, not chemical identity or a promise
that different floating-point conversion histories produce the same hash.

A cluster is geometric grouping. Its size is not probability, free energy,
independent sampling, affinity or biological validation.

:::{seealso}
{ref}`Tutorial_Get_pairwise_rmsd` and {ref}`Cookbook_ClusteringPoses` show how to
retain coordinate provenance and original member labels.
:::
